In [ ]:
import torch
from tqdm import tqdm
from nnsight import LanguageModel

torch.set_grad_enabled(False)

In [ ]:
mixtral = LanguageModel("mistralai/Mistral-7B-Instruct-v0.2", device_map="cuda:0", dispatch=True)

print("Loaded Mixtral in 4bit")

In [ ]:
past_key_values = None

In [ ]:
conversation_state = None  # This will hold past_key_values
conversation_history_ids = None  # This will store the tokenized conversation history

In [ ]:
def gen(
    model, 
    messages, 
    remote=False, max_new_tokens=300):
    prompt = model.tokenizer.apply_chat_template(messages, return_tensors="pt")

    sampling_kwargs = {
        "do_sample": True,
        "top_p": 0.3,
        "repetition_penalty": 1.1,
    }
        
    with model.generate(prompt, max_new_tokens=max_new_tokens, remote=remote, scan=False, validate=False, **sampling_kwargs):
        tokens = model.generator.output.save()
    
    new_tokens = tokens[0][len(prompt[0]):]
    torch.cuda.empty_cache()
    return model.tokenizer.decode(new_tokens)

In [ ]:
def gen(
    
)

In [ ]:
def append_and_generate(user_input, conversation_history_ids, conversation_state):
    # Tokenize the new user input and append to the conversation history
    new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
    bot_input_ids = torch.cat([conversation_history_ids, new_user_input_ids], dim=-1) if conversation_history_ids is not None else new_user_input_ids
    
    # Generate a response using the updated conversation history
    model_output = model.generate(bot_input_ids, max_length=1000, pad_token_id=tokenizer.eos_token_id,
                                  past_key_values=conversation_state, use_cache=True)
    
    # Update the conversation history with the generated response
    conversation_history_ids = model_output
    # Extract the `past_key_values` from the model output to use in the next turn
    conversation_state = model._reorder_cache(model_output.past_key_values)

    # Decode the generated response
    response = tokenizer.decode(model_output[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    
    return response, conversation_history_ids, conversation_state


In [ ]:
user_input = "Hello, how are you?"  # Example user input
response, conversation_history_ids, conversation_state = append_and_generate(user_input, conversation_history_ids, conversation_state)
print("Bot:", response)
# Continue the conversation by calling append_and_generate with new user inputs


In [ ]:
if past_key_values is None:
    with mixtral.trace("hello", scan=False, validate=False, use_cache=True) as trace:
        out = mixtral.output.save()

    past_key_values = out.value.past_key_values
else:
    with mixtral.trace("hello, my name is", scan=False, validate=False, use_cache=True, past_key_values=past_key_values) as trace:
        out = mixtral.output.save()

    past_key_values = out.value.past_key_values